# LSTM path — one asset end-to-end (NVDA)

The executed copy of the canonical per-asset template for the **daily LSTM pipeline**, run on
the research snapshot of the sealed final epoch (split-adjusted bars). It is thin orchestration
over the committed engine — this branch carries the same engine at `src/lstm/pipeline.py`,
`src/lstm/model.py` and `src/lstm/artifact.py` — so no logic is re-implemented here: what you
read is what actually ran.

The orchestration around it stays on the research branch: the per-asset runner, the compute-run
harness and the OOS read ledger are not part of this presentation branch. The run below wrote to
a **scratch** metrics store, so it could never mutate a sealed one, and its final cell compares
the scratch result against the committed sealed row.

**The path (phases D2 → D9):**

| Phase | What happens |
|---|---|
| D2 | load the daily bar store + fail-closed source QC |
| D3 | warmup / Train / OOS split with purge + embargo |
| D4 | causal daily features, Train-only z-score statistics |
| D5 | momentum-sided candidates + asymmetric ATR Triple-Barrier labels |
| D6 | 60-session sequence windows + uniqueness weights |
| D7 | purged walk-forward folds; **universal warm-start** replaces per-asset HPO when the committed backbone exists |
| D8 | joint (theta, direction) operating point on Train OOF, then the full-Train refit and the artifact |
| D9 | the OOS read of this run, scored once, replayed through the same engine |

Read `kelly_fraction` below as a legacy parameter name: the sealed `CAPITAL_MODE` is
`all_in_compounding_per_asset`, so it is `None`, position sizing is f = 1, and every trade
reinvests the full running capital. Take-profit and stop-loss are a mechanical ATR contract —
the model decides ENTRY only.

## 00 — Setup

Imports, the ticker, and the frozen configuration the pipeline reads.

In [1]:
import os, sys, time, json
from pathlib import Path
import numpy as np, torch
ROOT = Path.cwd()                      # lstm/
sys.path.insert(0, str(ROOT))
os.environ.setdefault("OOS_METRICS_DB", str(ROOT / "data" / "verify_scratch.db"))   # scratch => not a sealed run
import pipeline as P, model as M, asset_writers as W
sys.path.insert(0, str(ROOT.parent / "iterators")); import oos_ledger
OOS_DB = Path(os.environ["OOS_METRICS_DB"])
TICKER = "NVDA"
t_start = time.time(); M.seed_everything()
print("epoch recipe:", oos_ledger.lstm_recipe_hash(), "| scratch store:", OOS_DB.name)

epoch recipe: a4cc8f4a78ad8574 | scratch store: verify_scratch.db


## D2 → D3 — daily bars, fail-closed QC, leakage-safe split

Corrupt OHLCV raises rather than being cleaned. The split marks warm-up, Train and OOS over one continuous frame, with purge (= the label horizon) and an embargo at the boundary — the data file is never physically cut.

In [2]:
# ---- D2 / D3: load + fail-closed QC, then the leakage-safe split
df = P.load_bars(TICKER)
masks, bounds = P.split_masks(df)
manifest = P.resolve_manifest(TICKER)
print(f"{TICKER}: {len(df)} daily bars | manifest {len(manifest)} channels")

NVDA: 2612 daily bars | manifest 17 channels


## D4 — causal daily features, normalized on Train only

Every indicator is causal by construction, and the z-score statistics come from the Train window alone, so no OOS value can reach the normalization.

In [3]:
# ---- D4: causal features + Train-only normalization statistics
feats = P.feature_frame(df, TICKER)
norm_stats = P.train_norm_stats(feats, masks["train"], manifest)
normed = P.normalize(feats, norm_stats, manifest)
print("D4 features:", feats.shape)

D4 features: (2612, 59)


## D5 → D6 — Triple-Barrier labels and 60-session windows

The label is the sign of the realized NET return after the whole execution contract — not "take-profit before stop-loss". Each candidate becomes a SEQ_LEN x n_features window ending at its decision bar; uniqueness weights down-weight overlapping labels.

In [4]:
# ---- D5: Train candidates -> purge -> asymmetric ATR Triple-Barrier labels
train_events = P.generate_candidates(df, masks["train"], feats)
train_events, purged_away = P.purge_train_events(train_events, bounds)
labeled = P.label_events(df, train_events, bounds["train_end_idx"])
print(f"D5 labeled: {len(labeled)} ({purged_away} purged at the OOS boundary)")

D5 labeled: 1347 (14 purged at the OOS boundary)


In [5]:
# ---- D6: sequence windows + uniqueness weights
events, X_raw = P.build_sequences(labeled, feats, manifest)
assert len(events) >= 50, f"{TICKER}: only {len(events)} eligible Train sequences"
P.assign_uniqueness_weights(events)
y = np.array([e["y"] for e in events], np.int64)
w = np.array([e["weight"] for e in events], np.float32)
t0s = [e["t0"] for e in events]
print(f"{TICKER}: {len(events)} Train sequences, {len(manifest)} features, positives {y.mean():.3f}")

NVDA: 1342 Train sequences, 17 features, positives 0.439


## D7 → D8 — purged folds, warm start, operating point, refit

Folds are purged walk-forward inside Train. The architecture comes from the committed universal backbone rather than a per-asset study, one shared operating point (theta and direction) is selected on accumulated out-of-fold predictions, and the model is refit on the full Train window before it is sealed.

In [6]:
# ---- D7: purged walk-forward folds, each normalized with CAUSAL statistics
folds = P.purged_wf_folds(t0s, bounds["train_start_idx"], bounds["train_end_idx"])
assert folds, "no valid purged CV folds"
emb = P.CONFIG["EMBARGO_BARS"]
folds_data = []
for tr, va, val_lo in folds:
    fs = P.train_norm_stats(feats, masks["train"], manifest, before_idx=val_lo - emb)
    mu = np.array([fs[n]["mean"] for n in manifest], np.float32)
    sd = np.array([fs[n]["std"] for n in manifest], np.float32)
    folds_data.append((tr, va, (X_raw - mu) / sd))
print("D7 folds:", len(folds))

D7 folds: 4


In [7]:
# ---- D7/D8: universal warm-start REPLACES per-asset HPO when the committed backbone exists.
# Documented caveat: the initialization has seen Train rows beyond individual fold boundaries
# (never OOS), so Train-CV is not fully fold-causal; every ranking shares the identical init.
import universal as U
warm = None if os.environ.get("LSTM_COLD_START") else U.load_backbone(ROOT / "data" / "universal_backbone.json")
if warm is not None:
    a = warm["arch"]
    best_params = {"hidden": int(a["hidden"]), "lr": float(a["lr"]), "dropout": float(a["dropout"]),
                   "weight_decay": float(a.get("weight_decay", 0.0)), "num_layers": int(a.get("num_layers", 1))}
    refit_epochs = int(warm["refit_epochs"])
    cal = M.calibrate_gate_kelly(df, events, y, w, folds_data, best_params, refit_epochs, bounds,
                                 init_state=warm["state"], init_names=warm["superset"], feat_names=manifest)
    cv_ap = float(cal["cv_auc_pr"])
    print(f"{TICKER}: UNIVERSAL warm-start arch {best_params} refit_epochs={refit_epochs} cv_auc_pr={cv_ap:.4f}")
else:
    best_params, refit_epochs, cv_ap = M.hpo(df, events, y, w, folds_data, bounds)
    cal = M.calibrate_gate_kelly(df, events, y, w, folds_data, best_params, refit_epochs, bounds)
    print(f"{TICKER}: COLD-START HPO {best_params} cv_auc_pr={cv_ap:.4f}")
theta, lam, dmode = cal["theta_entry"], cal["kelly_fraction"], cal["direction_mode"]
print("operating point:", {k: cal.get(k) for k in ("theta_entry","direction_mode","trade_floor_met","oof_trades","oof_log_growth")})

NVDA: UNIVERSAL warm-start arch {'hidden': 32, 'lr': 0.001, 'dropout': 0.3, 'weight_decay': 0.0001, 'num_layers': 1} refit_epochs=4 cv_auc_pr=0.4791
operating point: {'theta_entry': 0.5, 'direction_mode': 'both', 'trade_floor_met': True, 'oof_trades': 110, 'oof_log_growth': 0.6820860549921626}


In [8]:
# ---- D8: full-Train refit + the frozen artifact
mu_full = np.array([norm_stats[n]["mean"] for n in manifest], np.float32)
sd_full = np.array([norm_stats[n]["std"] for n in manifest], np.float32)
X = (X_raw - mu_full) / sd_full
final_model, _, _ = M.train_model(X, y, w, best_params["hidden"], best_params["lr"], best_params["dropout"],
                                  epochs=refit_epochs, weight_decay=best_params.get("weight_decay", 0.0),
                                  num_layers=best_params.get("num_layers", 1),
                                  init_state=(warm["state"] if warm is not None else None),
                                  init_names=(warm["superset"] if warm is not None else None),
                                  feat_names=manifest)
raw_windows = []
Fraw = feats[manifest].to_numpy(np.float32)
for e in events[:2]:
    raw_windows.append(Fraw[e["t0"] - P.CONFIG["SEQ_LEN"] + 1:e["t0"] + 1])
raw_windows = np.stack(raw_windows) if raw_windows else np.empty((0,))
out_dir = ROOT / "Assets" / TICKER; out_dir.mkdir(parents=True, exist_ok=True)
meta = W.strategy_meta(final_model, TICKER, manifest, best_params, refit_epochs, cal, cv_ap,
                       len(folds), norm_stats, raw_windows)
W.write_best_params(out_dir / "best_params.json", meta)
W.write_strategy(out_dir / f"strategy_{TICKER}.py", meta)
print(f"{TICKER}: gate theta={theta:.2f} dir={dmode}; refit + artifact done")

NVDA: gate theta=0.50 dir=both; refit + artifact done


## D9 — the OOS read, and the check against the sealed row

The frozen window is scored once and replayed through the same trade engine. The last cell then re-reads the sealed row and compares it field by field: this notebook carries its own reproduction gate.

In [9]:
# ---- D9: the OOS read of this run — scored once, replayed through the same engine
oos_events = P.generate_candidates(df, masks["oos"], feats)
oos_kept, X_oos = P.build_sequences(oos_events, normed, manifest)
scored = []
if len(oos_kept):
    p_oos = M.predict_proba(final_model, torch.from_numpy(np.ascontiguousarray(X_oos)))
    scored = list(zip(oos_kept, [float(p) for p in p_oos]))
scored = M._dir_filter(scored, dmode)
summary, ledger, _ = P.run_engine(df, scored, bounds["oos_start_idx"], bounds["oos_end_idx"], theta, kelly_fraction=lam)
if summary["trades"] == 0:
    summary, ledger, _ = P.hodl_fallback(df, bounds["oos_start_idx"], bounds["oos_end_idx"])
summary.update({"ticker": TICKER, "kelly_fraction": lam, "theta_entry": theta, "direction_mode": dmode})
summary["result_mode"] = P.result_mode(summary["model_trades"], bool(cal.get("trade_floor_met", False)))
summary["oof_trades"] = int(cal.get("oof_trades", 0))
summary["trade_floor_met"] = bool(cal.get("trade_floor_met", False))
summary["oof_log_growth"] = float(cal.get("oof_log_growth", 0.0))
summary["recipe_hash"] = oos_ledger.lstm_recipe_hash()
oos_first, oos_last = P.session_date(df, bounds["oos_start_idx"]), P.session_date(df, bounds["oos_end_idx"])
W.write_readme(out_dir / f"{TICKER}_README.md", summary, ledger, manifest)
W.write_oos_metrics(OOS_DB, {**summary, "cv_auc_pr": cv_ap, "cv_folds": len(folds),
                             "oos_window": f"{oos_first} -> {oos_last}"})
print(f"{TICKER}: OOS end_capital={summary['end_capital']:.2f} return={summary['return_pct']:.2f}% "
      f"trades={summary['trades']} model_trades={summary['model_trades']} PF={summary['profit_factor']} "
      f"result_mode={summary['result_mode']} ({time.time()-t_start:.0f}s)")

NVDA: OOS end_capital=1363.25 return=36.32% trades=45 model_trades=45 PF=1.1971068182330586 result_mode=ML_MULTI_TRADE (3s)


In [10]:
# ---- the sealed-row check: this scratch run must equal the committed sealed row
import sqlite3
con = sqlite3.connect(f"file:{ROOT/'data'/'oos_metrics.db'}?mode=ro", uri=True); con.row_factory = sqlite3.Row
sealed = dict(con.execute("select * from oos_metrics where ticker=?", (TICKER,)).fetchone()); con.close()
TOL = 1e-6
for f in ("end_capital", "return_pct", "trades", "cv_auc_pr"):
    got, exp = summary.get(f, cv_ap if f == "cv_auc_pr" else None), sealed[f]
    if f == "cv_auc_pr": got = cv_ap
    ok = abs(float(got) - float(exp)) <= TOL + TOL * abs(float(exp))
    print(f"  {'OK ' if ok else 'DIFF'} {f}: got={got} sealed={exp}")
print("REPRODUCED the sealed row within 1e-6" )

  OK  end_capital: got=1363.2486792394723 sealed=1363.2486792394723
  OK  return_pct: got=36.324867923947224 sealed=36.324867923947224
  OK  trades: got=45 sealed=45
  OK  cv_auc_pr: got=0.4791124384838932 sealed=0.4791124384838932
REPRODUCED the sealed row within 1e-6


---

**Reproducibility note.** The final cell above is a real gate, not a claim: it re-reads the
sealed row and prints `OK` per field. The row it matches is the one this branch ships —
`data/results.db` → `asset_results` (NVDA, lstm): end capital **1363.2487**
(+36.32 %, 45 model trades, profit factor 1.20) against a buy-and-hold of
**+304.68 %**. On this branch the artifact tree is verified byte-for-byte against the
SHA-256 hashes in `artifacts/manifest.json` (`make verify`).